# Conceptual Dependency Representation from English Sentences

A rule-based Python program that extracts participants and maps simple English action verbs to CD primitives.

**Environment:** Google Colab / Jupyter Notebook  
**Language:** Python

## 1. Aim
Write a Python program that accepts a simple English sentence, identifies the actor, action, object, and receiver, and maps the verb to a Conceptual Dependency (CD) primitive.

## 2. Theory
Conceptual Dependency is a knowledge representation approach that expresses the meaning of natural-language actions using a small set of conceptual primitives. The same primitive can represent different surface verbs when they express a similar underlying action.

This practical implements a small, rule-based subset:
- **ATRANS:** transfer of possession or ownership.
- **PTRANS:** physical movement of an object or person.
- **MTRANS:** transfer of information or communication.

The program uses simple sentence patterns and verb dictionaries. It is an educational demonstration, not a general-purpose natural-language parser.

## 3. Import required library

In [ ]:
import re
from pprint import pprint
print("Conceptual Dependency parser ready.")

## 4. Define CD primitive verb groups
Verb groups include common past and present forms. Extend these lists to support more sentence patterns.

In [ ]:
CD_VERBS = {
    "ATRANS": {
        "give", "gives", "gave", "send", "sends", "sent",
        "hand", "hands", "handed", "sell", "sells", "sold",
        "lend", "lends", "lent", "offer", "offers", "offered"
    },
    "PTRANS": {
        "go", "goes", "went", "move", "moves", "moved",
        "travel", "travels", "travelled", "walk", "walks", "walked",
        "carry", "carries", "carried", "bring", "brings", "brought",
        "take", "takes", "took"
    },
    "MTRANS": {
        "tell", "tells", "told", "say", "says", "said",
        "inform", "informs", "informed", "teach", "teaches", "taught",
        "explain", "explains", "explained", "show", "shows", "showed"
    }
}

verb_to_primitive = {
    verb: primitive
    for primitive, verbs in CD_VERBS.items()
    for verb in verbs
}

print("Supported verb count:", len(verb_to_primitive))

## 5. Define a simple sentence parser
The parser supports patterns such as `John gave Mary a book`, `Mary sent a letter to John`, `John went to school`, and `Mary told John a story`. It uses a small set of prepositions and grammar patterns.

In [ ]:
def clean_words(sentence):
    return re.findall(r"[A-Za-z']+", sentence.lower())

def title_case_phrase(words):
    return " ".join(words).strip().title()

def parse_cd(sentence):
    words = clean_words(sentence)
    if len(words) < 2:
        return {"error": "Please enter a sentence with at least an actor and an action."}

    # Locate a supported verb. This simple parser expects a single main verb.
    verb_index = next((i for i, word in enumerate(words) if word in verb_to_primitive), None)
    if verb_index is None or verb_index == 0:
        return {"error": "No supported action verb was found, or the actor is missing."}

    actor = title_case_phrase(words[:verb_index])
    verb = words[verb_index]
    primitive = verb_to_primitive[verb]
    rest = words[verb_index + 1:]

    result = {
        "primitive": primitive,
        "actor": actor,
        "verb": verb,
        "object": None,
        "receiver": None,
        "from": None,
        "to": None
    }

    if not rest:
        return {"error": "The sentence is missing information after the action verb."}

    # ATRANS / MTRANS: e.g. John gave Mary a book; Mary told John a story.
    if primitive in {"ATRANS", "MTRANS"}:
        if "to" in rest:
            idx = rest.index("to")
            receiver_words = rest[idx + 1:]
            object_words = rest[:idx]
            result["object"] = title_case_phrase(object_words) or None
            result["receiver"] = title_case_phrase(receiver_words) or None
        elif len(rest) >= 2:
            # Direct-object + receiver + remaining object pattern:
            # gave Mary a book -> receiver=Mary, object=a book
            result["receiver"] = title_case_phrase(rest[0:1])
            result["object"] = title_case_phrase(rest[1:])
        else:
            result["object"] = title_case_phrase(rest)
        if primitive == "ATRANS":
            result["from"] = actor
            result["to"] = result["receiver"]
        else:
            result["from"] = actor
            result["to"] = result["receiver"]

    # PTRANS: e.g. John went to school; John carried a box to Mary.
    elif primitive == "PTRANS":
        if "to" in rest:
            idx = rest.index("to")
            before_to = rest[:idx]
            destination = rest[idx + 1:]
            result["to"] = title_case_phrase(destination) or None
            if before_to:
                result["object"] = title_case_phrase(before_to)
            else:
                result["object"] = actor
        elif "from" in rest:
            idx = rest.index("from")
            result["object"] = title_case_phrase(rest[:idx]) or actor
            result["from"] = title_case_phrase(rest[idx + 1:]) or None
        else:
            result["object"] = title_case_phrase(rest)

    return result

def display_cd(sentence):
    parsed = parse_cd(sentence)
    print("\nInput:", sentence)
    print("\nConceptual Dependency Representation")
    if "error" in parsed:
        print("Error:", parsed["error"])
        return parsed

    print("Action:", parsed["primitive"])
    print("Actor:", parsed["actor"])
    print("Object:", parsed["object"] or "Not identified")
    if parsed["receiver"]:
        print("Receiver:", parsed["receiver"])
    if parsed["from"]:
        print("From:", parsed["from"])
    if parsed["to"]:
        print("To:", parsed["to"])
    return parsed

## 6. Required example: ATRANS

In [ ]:
example = "John gave Mary a book"
result = display_cd(example)

# Verify the expected interpretation.
assert result["primitive"] == "ATRANS"
assert result["actor"] == "John"
assert result["object"] == "A Book"
assert result["receiver"] == "Mary"
assert result["from"] == "John"
assert result["to"] == "Mary"
print("\nExample checks passed.")

## 7. Test PTRANS and MTRANS examples

In [ ]:
test_sentences = [
    "John went to school",
    "Alice carried a box to the office",
    "Mary told John a story",
    "Ravi sent a letter to Sita"
]

for sentence in test_sentences:
    display_cd(sentence)
    print("-" * 45)

## 8. Accept user input
Run this cell in Colab or a local Jupyter session to enter your own sentence. The example is used as the default when input is left blank.

In [ ]:
user_sentence = input("Enter a simple English sentence (press Enter for example): ").strip()
if not user_sentence:
    user_sentence = "John gave Mary a book"

display_cd(user_sentence)

## 9. Observations
- The verb determines the CD primitive according to the defined verb dictionary.
- The parser extracts a simple actor from the words before the verb.
- The remaining words are interpreted using a few fixed patterns to identify an object, receiver, source, or destination.
- The required sentence `John gave Mary a book` is represented as ATRANS, with John as actor and source, Mary as receiver, and book as the transferred object.

## 10. Limitations
1. The program handles only a limited set of English verbs and sentence structures.
2. It does not perform full grammatical parsing, named-entity recognition, or word-sense disambiguation.
3. Ambiguous sentences and complex clauses may be parsed incorrectly.
4. A larger system would use a dependency parser and semantic role labelling before constructing CD structures.

## 11. Conclusion
A rule-based Python program was implemented to convert selected simple English sentences into Conceptual Dependency representations. It identifies the actor and action, extracts available objects and participants, and maps supported verbs to ATRANS, PTRANS, or MTRANS.